# Hierarchical vs. Flat Truncation Summarization for Long-Form Serialized Fiction




## 0. Setup

Note on versions: this notebook targets **transformers v5**. If `evaluate` or `rouge_score`
are missing, install them first.

In [ ]:
# If running fresh, uncomment:
!pip install -q transformers datasets evaluate rouge_score scipy accelerate
# torch must beCUDA build:
# !pip install torch --index-url https://download.pytorch.org/whl/cu121

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
from scipy.stats import wilcoxon
import transformers

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
e:\pyth\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.3.0
torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3050


## 1. Load data



In [ ]:
N_CHAPTERS = 40          # Chanege to 8 if running small exp
MIN_CHAPTER_WORDS = 1500  
RANDOM_SEED = 42

chapter_ds = load_dataset("ubaada/booksum-complete-cleaned", "chapters")["test"]

def has_valid_summary(example):
    return (
        isinstance(example["summary"], list)
        and len(example["summary"]) > 0
        and len(example["text"].split()) >= MIN_CHAPTER_WORDS
    )

filtered = chapter_ds.filter(has_valid_summary)
filtered = filtered.shuffle(seed=RANDOM_SEED).select(range(min(N_CHAPTERS, len(filtered))))

print(f"Selected {len(filtered)} chapters\n")
for i, ex in enumerate(filtered):
    print(f"{i}: {ex['book_title']} — {len(ex['text'].split())} words")


Selected 40 chapters

0: Tess of the D'Urbervilles — 3920 words
1: Sense and Sensibility — 2923 words
2: The Red and the Black — 2554 words
3: The Picture of Dorian Gray — 2819 words
4: The Brothers Karamazov — 4389 words
5: Lord Jim — 4328 words
6: Sense and Sensibility — 2474 words
7: The Brothers Karamazov — 3469 words
8: Uncle Vanya — 4803 words
9: The Brothers Karamazov — 2641 words
10: Far from the Madding Crowd — 2020 words
11: Sense and Sensibility — 1670 words
12: The Picture of Dorian Gray — 10520 words
13: The Autobiography of an Ex-Colored Man — 9043 words
14: Sense and Sensibility — 1504 words
15: The Brothers Karamazov — 6242 words
16: The White Devil — 3063 words
17: Lord Jim — 2192 words
18: The Prince — 2734 words
19: The Red and the Black — 8779 words
20: The Red and the Black — 1872 words
21: The Brothers Karamazov — 3375 words
22: The Brothers Karamazov — 3099 words
23: Far From the Madding Crowd — 2860 words
24: The Red and the Black — 2908 words
25: The Brothers K

## 2. Load the model 

In [3]:
MODEL_NAME = "facebook/bart-large-cnn"
MAX_INPUT_TOKENS = 1024

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()

print("Model loaded on:", device)
print("Encoder max position embeddings:", model.config.max_position_embeddings)


Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:01<00:00, 385.01it/s]


Model loaded on: cuda
Encoder max position embeddings: 1024


In [ ]:
@torch.no_grad()
def summarize_text(text, max_new_tokens=180, min_new_tokens=50, num_beams=4):
    """ Input is truncated to the model's 1024-token limit."""
    inputs = tokenizer(
        text,
        max_length=MAX_INPUT_TOKENS,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# hope this work

print(summarize_text(filtered[0]["text"][:4000], max_new_tokens=80, min_new_tokens=30))


Both `max_new_tokens` (=80) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The story of a newly-married couple who fall in love is told in the novel "Tess of the D'Urbervilles" The novel is about a young woman who falls in love with a man she once knew. The couple's love story is told through the eyes of the woman's daughter.


## 3. Chunking work

In [ ]:
def chunk_text_by_tokens(text, tokenizer, max_tokens=1000):
    sentences = [s.strip() for s in text.replace("\n", " ").split(". ") if s.strip()]
    chunks, current, current_len = [], [], 0

    for sent in sentences:
        sent_len = len(tokenizer.encode(sent, add_special_tokens=False))
        if sent_len > max_tokens:
            if current:
                chunks.append(". ".join(current) + ".")
                current, current_len = [], 0
            chunks.append(sent)
            continue

        if current_len + sent_len > max_tokens and current:
            chunks.append(". ".join(current) + ".")
            current, current_len = [], 0

        current.append(sent)
        current_len += sent_len

    if current:
        chunks.append(". ".join(current) + ".")
    return chunks


# testing if ts even work
test_chunks = chunk_text_by_tokens(filtered[0]["text"], tokenizer)
print(f"First chapter split into {len(test_chunks)} chunks")
print("Chunk token lengths:", [len(tokenizer.encode(c, add_special_tokens=False)) for c in test_chunks])


First chapter split into 6 chunks
Chunk token lengths: [950, 1022, 996, 1011, 1004, 335]


## 4. Define the two conditions and run them

In [ ]:
def flat_truncate_summary(chapter_text):
    return summarize_text(chapter_text, max_new_tokens=180, min_new_tokens=50)


def hierarchical_summary(chapter_text, tokenizer):
    chunks = chunk_text_by_tokens(chapter_text, tokenizer, max_tokens=1000)
    chunk_summaries = [
        summarize_text(c, max_new_tokens=100, min_new_tokens=30) for c in chunks
    ]
    combined = " ".join(chunk_summaries)
    final = summarize_text(combined, max_new_tokens=180, min_new_tokens=50)
    return final, len(chunks)


results = []
for i, ex in enumerate(filtered):
    chapter_text = ex["text"]
    reference = ex["summary"][0]["text"]

    flat_sum = flat_truncate_summary(chapter_text)
    hier_sum, n_chunks = hierarchical_summary(chapter_text, tokenizer)

    results.append({
        "idx": i,
        "book_title": ex["book_title"],
        "chapter_words": len(chapter_text.split()),
        "n_chunks": n_chunks,
        "reference": reference,
        "flat_summary": flat_sum,
        "hierarchical_summary": hier_sum,
    })
    print(f"[{i+1}/{len(filtered)}] {ex['book_title']} — {n_chunks} chunks")

results_df = pd.DataFrame(results)
results_df[["book_title", "chapter_words", "n_chunks"]]


Both `max_new_tokens` (=180) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=50) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

[1/40] Tess of the D'Urbervilles — 6 chunks


Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

[2/40] Sense and Sensibility — 4 chunks


Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

...

Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

[39/40] Sense and Sensibility — 3 chunks


Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

[40/40] The School for Scandal — 11 chunks


,book_title,chapter_words,n_chunks
0,Tess of the D'Urbervilles,3920,6
1,Sense and Sensibility,2923,4
2,The Red and the Black,2554,4
3,The Picture of Dorian Gray,2819,4
4,The Brothers Karamazov,4389,6
5,Lord Jim,4328,6
6,Sense and Sensibility,2474,4
7,The Brothers Karamazov,3469,5
8,Uncle Vanya,4803,7
9,The Brothers Karamazov,2641,4


Output testing 

In [ ]:
row = results_df.iloc[0]
print("=== REFERENCE ===\n", row["reference"][:800], "\n")
print("=== FLAT TRUNCATION ===\n", row["flat_summary"], "\n")
print("=== HIERARCHICAL ===\n", row["hierarchical_summary"])


=== REFERENCE ===
 Angel arises at dawn; the neighboring cottager's wife knocks on the door, but he sends her away because her presence is awkward. Angel prepares breakfast, and the two behave civilly to one another, although the pair are "but ashes of their former fires. Angel asks again if it is true, and he asks if the man is still in England. Tess says that he can get rid of her by divorcing her; her confession has given him adequate grounds for that. She tells him that she thought of putting an end to herself under the mistletoe, but did not because she felt it would cause scandal. Tess continues to do chores around the house for Angel while he visits a local miller, but he scolds her for behaving as a servant and not a wife. Tess breaks into tears, claiming that she had told him that she was not respec 

=== FLAT TRUNCATION ===
 The story of a newly-married couple who fall in love is told in the novel "Tess of the D'Urbervilles" The novel is about a young woman who falls in love 

## 5. ROUGE 

In [8]:
rouge = evaluate.load("rouge")

flat_scores = rouge.compute(
    predictions=results_df["flat_summary"].tolist(),
    references=results_df["reference"].tolist(),
    use_aggregator=False,
)
hier_scores = rouge.compute(
    predictions=results_df["hierarchical_summary"].tolist(),
    references=results_df["reference"].tolist(),
    use_aggregator=False,
)

for metric in ["rouge1", "rouge2", "rougeL"]:
    results_df[f"flat_{metric}"] = flat_scores[metric]
    results_df[f"hier_{metric}"] = hier_scores[metric]

results_df[[
    "book_title", "n_chunks",
    "flat_rouge1", "hier_rouge1",
    "flat_rouge2", "hier_rouge2",
    "flat_rougeL", "hier_rougeL",
]].round(4)


,book_title,n_chunks,flat_rouge1,hier_rouge1,flat_rouge2,hier_rouge2,flat_rougeL,hier_rougeL
0,Tess of the D'Urbervilles,6,0.1448,0.1737,0.0162,0.0053,0.0912,0.0895
1,Sense and Sensibility,4,0.2172,0.2162,0.0274,0.0364,0.1267,0.1351
2,The Red and the Black,4,0.1023,0.1762,0.0188,0.0267,0.0651,0.1145
3,The Picture of Dorian Gray,4,0.1117,0.0995,0.0051,0.0053,0.0863,0.0681
4,The Brothers Karamazov,6,0.1747,0.1468,0.0264,0.0000,0.1135,0.1009
5,Lord Jim,6,0.0721,0.0959,0.0150,0.0271,0.0498,0.0640
6,Sense and Sensibility,4,0.1714,0.1788,0.0000,0.0000,0.1029,0.1229
7,The Brothers Karamazov,5,0.1967,0.1923,0.0579,0.0388,0.1230,0.1231
8,Uncle Vanya,7,0.1308,0.1118,0.0169,0.0208,0.0802,0.0745
9,The Brothers Karamazov,4,0.1528,0.1132,0.0000,0.0127,0.0972,0.0755


## 6. Wilcoxon signed-rank test

In [9]:
print(f"n = {len(results_df)}\n")
for metric in ["rouge1", "rouge2", "rougeL"]:
    flat_vals = results_df[f"flat_{metric}"].values
    hier_vals = results_df[f"hier_{metric}"].values
    diff = hier_vals - flat_vals

    stat, p = wilcoxon(hier_vals, flat_vals)
    print(f"{metric.upper():8s}  flat={flat_vals.mean():.4f}  hier={hier_vals.mean():.4f}  "
          f"mean diff={diff.mean():+.4f}  W={stat:.1f}  p={p:.4f}")


n = 40

ROUGE1    flat=0.1431  hier=0.1411  mean diff=-0.0020  W=375.0  p=0.6463
ROUGE2    flat=0.0251  hier=0.0253  mean diff=+0.0002  W=337.0  p=0.4595
ROUGEL    flat=0.0912  hier=0.0900  mean diff=-0.0012  W=352.0  p=0.4437


## 7. Entity-retention check

The regex approach below is a crude proper-noun proxy. This is super crude because it only check for caps non-initialized entity

In [12]:
import re

STOPWORD_CAPS = {"The", "A", "An", "He", "She", "It", "They", "But", "And", "When", "This", "That"}

def extract_entities(text):
    """Crude proper-noun proxy: capitalized tokens not at sentence start."""
    words = text.split()
    candidates = set()
    prev_ends_sentence = True
    for w in words:
        cleaned = re.sub(r"[^A-Za-z]", "", w)
        if cleaned and cleaned[0].isupper() and not prev_ends_sentence:
            if cleaned not in STOPWORD_CAPS and len(cleaned) > 2:
                candidates.add(cleaned)
        prev_ends_sentence = w.endswith((".", "!", "?"))
    return candidates


def entity_recall(reference, generated):
    ref_ents = extract_entities(reference)
    if not ref_ents:
        return np.nan
    gen_ents = extract_entities(generated)
    return len(ref_ents & gen_ents) / len(ref_ents)


results_df["flat_entity_recall"] = results_df.apply(
    lambda r: entity_recall(r["reference"], r["flat_summary"]), axis=1)
results_df["hier_entity_recall"] = results_df.apply(
    lambda r: entity_recall(r["reference"], r["hierarchical_summary"]), axis=1)

print("Mean entity recall — flat:        ", round(results_df["flat_entity_recall"].mean(), 4))
print("Mean entity recall — hierarchical:", round(results_df["hier_entity_recall"].mean(), 4))

stat_e, p_e = wilcoxon(results_df["hier_entity_recall"], results_df["flat_entity_recall"])
print(f"Wilcoxon on entity recall: W={stat_e:.1f}, p={p_e:.4f}")

results_df[["book_title", "flat_entity_recall", "hier_entity_recall"]].round(4)

Mean entity recall — flat:         0.1599
Mean entity recall — hierarchical: 0.1464
Wilcoxon on entity recall: W=94.0, p=0.2912


,book_title,flat_entity_recall,hier_entity_recall
0,Tess of the D'Urbervilles,0.1429,0.0000
1,Sense and Sensibility,0.3750,0.2500
2,The Red and the Black,0.3333,0.3333
3,The Picture of Dorian Gray,0.2222,0.1111
4,The Brothers Karamazov,0.1667,0.3333
5,Lord Jim,0.0455,0.0000
6,Sense and Sensibility,0.0909,0.0909
7,The Brothers Karamazov,0.1667,0.1111
8,Uncle Vanya,0.0000,0.0000
9,The Brothers Karamazov,0.5000,0.5000


## 7. Save results

In [ ]:
results_df.to_csv("summarization_results.csv", index=False)
print("Saved summarization_results.csv")

report_table = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "Entity recall"],
    "Flat truncation": [
        results_df["flat_rouge1"].mean(),
        results_df["flat_rouge2"].mean(),
        results_df["flat_rougeL"].mean(),
        results_df["flat_entity_recall"].mean(),
    ],
    "Hierarchical": [
        results_df["hier_rouge1"].mean(),
        results_df["hier_rouge2"].mean(),
        results_df["hier_rougeL"].mean(),
        results_df["hier_entity_recall"].mean(),
    ],
}).round(4)

report_table


Saved summarization_results40chap.csv


,Metric,Flat truncation,Hierarchical
0,ROUGE-1,0.1431,0.1411
1,ROUGE-2,0.0251,0.0253
2,ROUGE-L,0.0912,0.0900
3,Entity recall,0.1599,0.1464
